# CR-FIQA 기반 조건부 threshold calibration

이 노트북은 완료된 SurvFace Step-4 run(기본값: ArcFace)을 **읽기 전용 입력**으로 사용해 다음 세 가지를 비교합니다.

1. `global_empirical`: 기존 calibration 전체에서 구한 전역 threshold
2. `global_safe`: calibration 내부 fit/safety 분할을 거친 보수적 전역 threshold
3. `fiqa_2bin_conservative_shrunk_safe`: CR-FIQA Low/High 그룹별 부분 풀링 + held-out safety threshold

Saliency의 연구상 위치는 바꾸지 않습니다.

- **1차 목적:** 원본 FR 모델의 공간적 인식 근거가 압축에 따른 embedding distortion, score/rank 변화, threshold crossing과 어떻게 연관되는지 분석
- **2차 목적:** FIQA만으로 설명되지 않는 threshold 불안정성을 saliency가 추가로 설명하는지 검증

현재 SurvFace saliency는 test probe에만 존재하므로, 1차 분석은 가능하지만 FIQA+Saliency threshold 학습은 calibration saliency가 확보될 때까지 누수 방지 게이트가 차단합니다. 기존 공통 orchestration/report 노트북은 수정하지 않습니다.

In [1]:
# 0. 프로젝트 경로와 공통 import
from __future__ import annotations

import gc
import json
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def find_project_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'research').is_dir() and (candidate / '.git').exists():
            return candidate
    raise RuntimeError('C:\\ronbun 프로젝트 루트를 찾지 못했습니다.')


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.evaluation import (
    assess_saliency_faithfulness_reliability,
    load_selected_faithfulness_artifacts,
    resolve_common_faithfulness_maximum_samples,
)
from research.experiments.fiqa_threshold_calibration import (
    assess_saliency_incremental_readiness,
    join_fiqa_score_artifacts,
    load_calibration_comparison_artifact,
    load_condition_score_artifact,
    load_saliency_primary_diagnostics,
    replay_survface_adc_condition_scores,
    run_global_vs_fiqa_calibration,
    write_calibration_comparison_artifact,
    write_condition_score_artifact,
)
from research.fiqa import (
    CRFIQA_VARIANTS,
    infer_cr_fiqa_scores,
    load_cr_fiqa,
    load_fiqa_score_artifact,
    materialize_aligned_bundle_score_artifact,
)
from research.runtime.hashing import sha256_file

PROJECT_ROOT

WindowsPath('C:/ronbun')

## 1. 실험 계약

- **주 실험:** CR-FIQA(S), Low/High 2분위, PQ `m=128`, ADC exhaustive search
- **민감도 분석:** 주 실험이 끝난 뒤 CR-FIQA(L)만 동일 설정으로 반복
- FIQA cutpoint와 모든 threshold는 calibration에서만 결정하며 test 결과를 보지 않습니다.
- ADC의 점수공간은 `negative_squared_l2_adc`이며 원본 cosine threshold를 재사용하지 않습니다.
- Low/High group 표본 부족 시 전역 threshold로 fallback하고, 충분한 경우에도 전역값과 부분 풀링합니다.
- `high_saliency`/`low_saliency` mask는 intervention/faithfulness 분석용입니다. `random`은 음성 대조군이며 threshold feature로 사용하지 않습니다.
- 완료 run과 checkpoint는 덮어쓰지 않습니다. 모든 결과 쓰기는 별도 플래그로 명시합니다.

In [2]:
# 2. 사용자가 조정하는 유일한 설정 셀
SURVFACE_RUN_CANDIDATES = {
    'arcface': PROJECT_ROOT / 'runs' / 'survface_20260902' / (
        '20260902-R001-61915edf_step4_survface_arcface-7972a704552df378345f'
    ),
    'adaface': PROJECT_ROOT / 'runs' / 'survface_20260830' / (
        '20260830-R001-ec6e5d4a_step4_survface_adaface-4df25b75e065b0b9ed43'
    ),
}
SOURCE_MODEL = 'arcface'
assert SOURCE_MODEL in SURVFACE_RUN_CANDIDATES, 'SOURCE_MODEL은 명시된 완료 run 후보여야 합니다.'
SOURCE_RUN_DIR = SURVFACE_RUN_CANDIDATES[SOURCE_MODEL]
ALIGNED_BUNDLE_DIR = PROJECT_ROOT / 'data' / 'interim' / 'step4' / 'survface' / 'aligned_112'
FIQA_CHECKPOINTS = {
    'S': PROJECT_ROOT / 'models' / 'fiqa' / 'CR-FIQA(S).pth',
    'L': PROJECT_ROOT / 'models' / 'fiqa' / 'CR-FIQA(L).pth',
}
# FIQA_VARIANT = 'S'
FIQA_VARIANT = 'L'
FIQA_BATCH_SIZE = 64
FIQA_SHARD_SIZE = 8192
MODEL_SMOKE_VARIANTS = ('S', 'L')
MODEL_SMOKE_SAMPLE_COUNT = FIQA_BATCH_SIZE

COMPRESSION_PROFILE = 'pq_512_m128_b8'
SEARCH_MODE = 'pq_adc_exhaustive'
TARGET_FPIRS = (0.01, 0.05, 0.10, 0.20, 0.30)
QUALITY_BIN_COUNT = 2
SHRINKAGE_STRENGTH = 200.0
MINIMUM_GROUP_NON_MATED = 100
SAFETY_FRACTION = 0.30
PARTITION_SEED = 8972

RESULT_ROOT = PROJECT_ROOT / 'results' / 'calibration'
VERIFY_CHECKPOINT_HASHES = True
OVERWRITE_OUTPUTS = False

# 긴 계산 및 쓰기는 아래 섹션별로 순서대로 실행하십시오.
# ==================== 모델 사전 점검 ====================
RUN_MODEL_SMOKE = False  # CR-FIQA S/L checkpoint GPU smoke test

# ==================== 1. FIQA 점수 생성 ====================
RUN_FIQA_INFERENCE = True  # 전체 aligned 얼굴의 CR-FIQA 점수 추론 실행
WRITE_FIQA_ARTIFACT = True  # 추론한 FIQA 점수를 artifact로 저장

# ==================== 2. PQ-m128 ADC 점수 재생성 ====================
RUN_SCORE_REPLAY = False  # calibration/test의 PQ-m128 ADC 점수 재계산
WRITE_SCORE_ARTIFACT = False  # 재계산한 ADC condition score를 저장

# ==================== 3. Global/FIQA threshold 비교 ====================
RUN_THRESHOLD_CALIBRATION = True  # Global과 FIQA 조건부 threshold 비교 실행
WRITE_CALIBRATION_ARTIFACT = True  # threshold 비교 결과를 artifact로 저장

# ==================== 4. Saliency 진단 ====================
RUN_PRIMARY_SALIENCY_SUMMARY = True
RUN_SALIENCY_READINESS_CHECK = True

assert FIQA_VARIANT in FIQA_CHECKPOINTS
assert QUALITY_BIN_COUNT == 2, '주 분석은 사전 지정된 Low/High 2분위를 사용합니다.'
for stage_name, run_enabled, write_enabled in (
    ('FIQA inference', RUN_FIQA_INFERENCE, WRITE_FIQA_ARTIFACT),
    ('ADC score replay', RUN_SCORE_REPLAY, WRITE_SCORE_ARTIFACT),
    ('threshold calibration', RUN_THRESHOLD_CALIBRATION, WRITE_CALIBRATION_ARTIFACT),
):
    if write_enabled and not run_enabled:
        raise ValueError(f'{stage_name}: WRITE=True이면 대응하는 RUN도 True여야 합니다.')

### 권장 실행 순서

노트북은 B → C → D 순서로 실행됩니다. 처음부터 모두 수행하려면 각 `RUN_*`/`WRITE_*` 플래그를 함께 `True`로 설정하고 위에서부터 한 번 실행할 수 있습니다. `OVERWRITE_OUTPUTS=False`이면 이미 완료된 artifact는 SHA-256 검증 후 재사용하고, 없는 artifact만 다음 순서로 생성하므로 재실행도 안전합니다.

1. `RUN_MODEL_SMOKE=True` — S/L checkpoint strict-load 및 production batch(기본 64장) GPU 추론
2. `RUN_FIQA_INFERENCE=True`, `WRITE_FIQA_ARTIFACT=True` — 전체 aligned bundle의 CR-FIQA(S) 점수 생성
3. `RUN_SCORE_REPLAY=True`, `WRITE_SCORE_ARTIFACT=True` — 저장되지 않았던 calibration PQ-ADC 검색만 재생하고 기존 threshold와 일치 여부 감사
4. `RUN_THRESHOLD_CALIBRATION=True`, `WRITE_CALIBRATION_ARTIFACT=True` — Global/FIQA 비교 산출
5. 필요할 때만 `FIQA_VARIANT='L'`로 바꾸어 민감도 분석 반복

실행이 중단되면 같은 설정으로 위에서부터 다시 실행합니다. 완료된 단계는 검증 후 건너뛰고, 중단된 FIQA shard부터 재개합니다. 기존 완료 artifact를 의도적으로 다시 만들 때만 `OVERWRITE_OUTPUTS=True`를 사용합니다.

In [3]:
# 3. 경량 preflight: 입력 존재, 모델 크기/hash, run 및 aligned contract
def read_json_object(path: Path) -> dict:
    payload = json.loads(path.read_text(encoding='utf-8'))
    if not isinstance(payload, dict):
        raise ValueError(f'JSON object가 아닙니다: {path}')
    return payload


if not SOURCE_RUN_DIR.is_dir() or not (SOURCE_RUN_DIR / 'COMPLETED').is_file():
    raise FileNotFoundError(f'완료된 source run이 없습니다: {SOURCE_RUN_DIR}')
source_manifest = read_json_object(SOURCE_RUN_DIR / 'run_manifest.json')
if source_manifest.get('status') != 'completed':
    raise ValueError('source run status가 completed가 아닙니다.')
if source_manifest.get('config', {}).get('dataset_id') != 'survface':
    raise ValueError('이 노트북의 첫 구현은 SurvFace 전용입니다.')
source_model_uid = str(source_manifest.get('config', {}).get('model_uid', ''))
if not source_model_uid:
    raise ValueError('source run에 model_uid가 없습니다.')

aligned_manifest = read_json_object(ALIGNED_BUNDLE_DIR / 'bundle_manifest.json')
contract = aligned_manifest.get('array_contract', {})
if not (
    contract.get('dtype') == 'uint8'
    and contract.get('layout') == 'nhwc'
    and contract.get('color_order') == 'rgb'
    and contract.get('image_size') == [112, 112]
):
    raise ValueError('CR-FIQA 입력은 uint8 NHWC RGB 112x112 aligned bundle이어야 합니다.')

checkpoint_rows = []
for variant, checkpoint in FIQA_CHECKPOINTS.items():
    spec = CRFIQA_VARIANTS[variant]
    if not checkpoint.is_file():
        raise FileNotFoundError(f'CR-FIQA({variant}) checkpoint가 없습니다: {checkpoint}')
    actual_sha256 = sha256_file(checkpoint) if VERIFY_CHECKPOINT_HASHES else None
    checkpoint_rows.append(
        {
            'variant': variant,
            'architecture': spec.architecture,
            'bytes_match': checkpoint.stat().st_size == spec.expected_bytes,
            'sha256_match': actual_sha256 == spec.expected_sha256 if actual_sha256 else 'not_checked',
            'model_uid': spec.model_uid,
            'license': spec.license_id,
        }
    )
checkpoint_preflight = pd.DataFrame(checkpoint_rows)
if not checkpoint_preflight['bytes_match'].all():
    raise ValueError('checkpoint byte size가 등록된 공식 파일과 다릅니다.')
if VERIFY_CHECKPOINT_HASHES and not checkpoint_preflight['sha256_match'].all():
    raise ValueError('checkpoint SHA-256이 등록된 공식 파일과 다릅니다.')

display(checkpoint_preflight)
display(
    pd.DataFrame(
        [{
            'source_run_id': source_manifest['run_id'],
            'dataset_id': source_manifest['config']['dataset_id'],
            'model_uid': source_manifest['config']['model_uid'],
            'aligned_rows': aligned_manifest.get('counts', {}).get('aligned'),
            'selected_fiqa_variant': FIQA_VARIANT,
        }]
    )
)

,variant,architecture,bytes_match,sha256_match,model_uid,license
0,S,iresnet50,True,True,cr-fiqa-s-b9f457a6f00e0363a0cf,CC-BY-NC-4.0
1,L,iresnet100,True,True,cr-fiqa-l-5fca24736e4f8df5fbfc,CC-BY-NC-4.0


,source_run_id,dataset_id,model_uid,aligned_rows,selected_fiqa_variant
0,20260902-R001-61915edf,survface,arcface-7972a704552df378345f,463341,L


## 4. 선택 단계 A — CR-FIQA S/L GPU smoke test

공식 구조와 state dict를 `strict=True`로 불러오며 CUDA가 없으면 CPU로 조용히 전환하지 않고 중단합니다. 점수는 sigmoid/min-max 변환 없이 checkpoint의 raw scalar 그대로 사용합니다.

In [4]:
smoke_summary = pd.DataFrame()
if RUN_MODEL_SMOKE:
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError('CUDA smoke test가 요청됐지만 torch.cuda.is_available()이 False입니다.')
    aligned_index = pd.read_csv(
        ALIGNED_BUNDLE_DIR / 'aligned_index.csv',
        nrows=MODEL_SMOKE_SAMPLE_COUNT,
        usecols=['sample_id', 'aligned_face_index'],
    )
    aligned_faces = np.load(
        ALIGNED_BUNDLE_DIR / 'aligned_faces.npy', mmap_mode='r', allow_pickle=False
    )
    face_indices = aligned_index['aligned_face_index'].to_numpy(dtype=np.int64)
    smoke_faces = np.asarray(aligned_faces[face_indices])
    smoke_rows = []
    for variant in MODEL_SMOKE_VARIANTS:
        torch.cuda.reset_peak_memory_stats()
        model, spec = load_cr_fiqa(
            FIQA_CHECKPOINTS[variant], variant=variant, device='cuda',
            verify_official_hash=True,
        )
        scores = infer_cr_fiqa_scores(
            model, smoke_faces, batch_size=MODEL_SMOKE_SAMPLE_COUNT, device='cuda'
        )
        smoke_rows.append(
            {
                'variant': variant,
                'model_uid': spec.model_uid,
                'sample_count': len(scores),
                'finite': bool(np.isfinite(scores).all()),
                'score_min': float(scores.min()),
                'score_max': float(scores.max()),
                'cuda_peak_bytes': int(torch.cuda.max_memory_allocated()),
                'device': torch.cuda.get_device_name(0),
            }
        )
        del model, scores
        gc.collect()
        torch.cuda.empty_cache()
    smoke_summary = pd.DataFrame(smoke_rows)
    display(smoke_summary)
else:
    display(Markdown('`RUN_MODEL_SMOKE=False`: checkpoint GPU smoke를 건너뜁니다.'))

`RUN_MODEL_SMOKE=False`: checkpoint GPU smoke를 건너뜁니다.

## 5. 선택 단계 B — 전체 SurvFace CR-FIQA score artifact

모든 aligned face에 scalar quality를 한 번만 계산해 `sample_id, fiqa_score, fiqa_model_uid` 형태로 저장합니다. 전체 입력 bundle과 checkpoint lineage를 manifest에 고정합니다. 각 8,192장 shard를 원자적으로 기록하므로 중단 후 동일 설정으로 다시 실행하면 완료 shard부터 재개합니다. 전체 추론을 요청할 때는 쓰기 플래그도 반드시 켜야 합니다.

In [5]:
selected_spec = CRFIQA_VARIANTS[FIQA_VARIANT]
FIQA_OUTPUT_DIR = RESULT_ROOT / 'fiqa_scores' / 'survface' / selected_spec.model_uid
fiqa_artifact = None
fiqa_stage_action = 'not_run'

if RUN_FIQA_INFERENCE:
    if FIQA_OUTPUT_DIR.exists() and not OVERWRITE_OUTPUTS:
        fiqa_artifact = load_fiqa_score_artifact(FIQA_OUTPUT_DIR)
        fiqa_stage_action = 'reused_verified'
        display(Markdown(f'완료된 FIQA artifact를 검증 후 재사용합니다: `{FIQA_OUTPUT_DIR}`'))
    else:
        if not WRITE_FIQA_ARTIFACT:
            raise ValueError('새 FIQA inference에는 WRITE_FIQA_ARTIFACT=True가 필요합니다.')
        model, loaded_spec = load_cr_fiqa(
            FIQA_CHECKPOINTS[FIQA_VARIANT], variant=FIQA_VARIANT, device='cuda',
            verify_official_hash=True,
        )
        fiqa_artifact = materialize_aligned_bundle_score_artifact(
            ALIGNED_BUNDLE_DIR,
            FIQA_OUTPUT_DIR,
            model=model,
            model_uid=loaded_spec.model_uid,
            checkpoint_sha256=model.cr_fiqa_checkpoint_sha256,
            variant=loaded_spec.variant,
            batch_size=FIQA_BATCH_SIZE,
            shard_size=FIQA_SHARD_SIZE,
            device='cuda',
            overwrite=OVERWRITE_OUTPUTS,
        )
        fiqa_stage_action = 'computed_written'
        del model
        gc.collect()
else:
    if FIQA_OUTPUT_DIR.is_dir():
        fiqa_artifact = load_fiqa_score_artifact(FIQA_OUTPUT_DIR)
        fiqa_stage_action = 'loaded_verified'

if fiqa_artifact is not None:
    expected_fiqa = {
        'dataset_id': 'survface',
        'fiqa_model_uid': selected_spec.model_uid,
        'checkpoint_sha256': selected_spec.expected_sha256,
    }
    mismatches = {
        key: (fiqa_artifact.manifest.get(key), expected)
        for key, expected in expected_fiqa.items()
        if fiqa_artifact.manifest.get(key) != expected
    }
    if mismatches:
        raise ValueError(f'FIQA artifact가 현재 설정과 다릅니다: {mismatches}')

if fiqa_artifact is None:
    display(Markdown(f'FIQA artifact 대기 중: `{FIQA_OUTPUT_DIR}`'))
else:
    display(pd.DataFrame([fiqa_artifact.manifest['score_summary']]))
    display(fiqa_artifact.scores.head())

,minimum,median,maximum,mean
0,-0.09688,0.562348,2.174285,0.626438


,sample_id,aligned_face_index,aligned_content_sha256,fiqa_score,fiqa_model_uid
0,survface:train:100:100_cam1_1,0,9e9420b9c0f32c3385712030093558439e48e4c419c448...,0.497116,cr-fiqa-l-5fca24736e4f8df5fbfc
1,survface:train:100:100_cam2_1,1,03f47e7734336617b9f673ae84056185b0d152d327983e...,0.516596,cr-fiqa-l-5fca24736e4f8df5fbfc
2,survface:train:100:100_cam3_1,2,c56689c351b775928adf7f1cc51137e7ac8b5e353dab2a...,0.399595,cr-fiqa-l-5fca24736e4f8df5fbfc
3,survface:train:100:100_cam4_1,3,9f44c44a593492e85960d7074c1151bdeb743ce037169f...,1.243854,cr-fiqa-l-5fca24736e4f8df5fbfc
4,survface:train:100:100_cam5_1,4,35ebf8e5bb132ff0339c886824ceab00180e765464708f...,1.075207,cr-fiqa-l-5fca24736e4f8df5fbfc


## 6. 선택 단계 C — calibration PQ-ADC score만 재생

완료된 Step-4 run은 test retrieval core를 보존했지만 calibration query의 개별 compressed score는 threshold 산출 뒤 저장하지 않았습니다. 따라서 이 단계는 기존 embedding과 frozen PQ codec을 재사용해 **calibration search만** 재생합니다. 재생한 점수로 기존 다섯 target FPIR threshold가 `1e-12` 이내에서 재현되지 않으면 artifact를 만들지 않습니다.

In [6]:
CONDITION_OUTPUT_DIR = (
    RESULT_ROOT / 'condition_scores' / str(source_manifest['run_id'])
    / f'{COMPRESSION_PROFILE}__{SEARCH_MODE}'
)
condition_tables = None
condition_stage_action = 'not_run'

if RUN_SCORE_REPLAY:
    if CONDITION_OUTPUT_DIR.exists() and not OVERWRITE_OUTPUTS:
        condition_tables = load_condition_score_artifact(CONDITION_OUTPUT_DIR)
        condition_stage_action = 'reused_verified'
        display(Markdown(f'완료된 condition score artifact를 검증 후 재사용합니다: `{CONDITION_OUTPUT_DIR}`'))
    else:
        if not WRITE_SCORE_ARTIFACT:
            raise ValueError('새 calibration score replay에는 WRITE_SCORE_ARTIFACT=True가 필요합니다.')
        in_memory_tables = replay_survface_adc_condition_scores(
            SOURCE_RUN_DIR, compression_profile=COMPRESSION_PROFILE, search_mode=SEARCH_MODE
        )
        condition_tables = write_condition_score_artifact(
            CONDITION_OUTPUT_DIR, in_memory_tables, overwrite=OVERWRITE_OUTPUTS
        )
        condition_stage_action = 'computed_written'
else:
    if CONDITION_OUTPUT_DIR.is_dir():
        condition_tables = load_condition_score_artifact(CONDITION_OUTPUT_DIR)
        condition_stage_action = 'loaded_verified'

if condition_tables is not None:
    expected_condition = {
        'source_run_id': str(source_manifest['run_id']),
        'model_uid': source_model_uid,
        'compression_profile': COMPRESSION_PROFILE,
        'search_mode': SEARCH_MODE,
    }
    mismatches = {
        key: (condition_tables.manifest.get(key), expected)
        for key, expected in expected_condition.items()
        if condition_tables.manifest.get(key) != expected
    }
    if mismatches:
        raise ValueError(f'condition artifact가 현재 설정과 다릅니다: {mismatches}')

if condition_tables is None:
    display(Markdown(f'condition score artifact 대기 중: `{CONDITION_OUTPUT_DIR}`'))
else:
    display(pd.DataFrame(condition_tables.manifest['global_threshold_reproduction']))
    display(
        pd.DataFrame(
            [{
                'condition_uid': condition_tables.condition_uid,
                'calibration_rows': len(condition_tables.calibration),
                'test_rows': len(condition_tables.test),
                'score_space': condition_tables.manifest['score_space'],
            }]
        )
    )

,target_fpir,persisted_threshold,reproduced_threshold,absolute_difference,exact_match
0,0.01,-0.355155,-0.355155,0.0,True
1,0.05,-0.425403,-0.425403,0.0,True
2,0.10,-0.470093,-0.470093,0.0,True
3,0.20,-0.532888,-0.532888,0.0,True
4,0.30,-0.584186,-0.584186,0.0,True


,condition_uid,calibration_rows,test_rows,score_space
0,compressed-scores-343aa7daf6fbb509924fd1b6,160408,182159,negative_squared_l2_adc


In [7]:
# 7. FIQA를 calibration/test query에 완전한 one-to-one으로 결합
calibration_with_fiqa = None
test_with_fiqa = None
if fiqa_artifact is not None and condition_tables is not None:
    calibration_with_fiqa, test_with_fiqa = join_fiqa_score_artifacts(
        condition_tables, fiqa_artifact
    )
    display(
        pd.DataFrame(
            [
                {
                    'split': 'calibration', 'rows': len(calibration_with_fiqa),
                    'fiqa_min': calibration_with_fiqa['fiqa_score'].min(),
                    'fiqa_median': calibration_with_fiqa['fiqa_score'].median(),
                    'fiqa_max': calibration_with_fiqa['fiqa_score'].max(),
                },
                {
                    'split': 'test', 'rows': len(test_with_fiqa),
                    'fiqa_min': test_with_fiqa['fiqa_score'].min(),
                    'fiqa_median': test_with_fiqa['fiqa_score'].median(),
                    'fiqa_max': test_with_fiqa['fiqa_score'].max(),
                },
            ]
        )
    )
else:
    display(Markdown('FIQA와 condition score artifact가 모두 준비되면 결합합니다.'))

,split,rows,fiqa_min,fiqa_median,fiqa_max
0,calibration,160408,-0.096880,0.560314,2.112883
1,test,182159,-0.044018,0.565888,2.174285


## 8. 선택 단계 D — Global 대 FIQA-conditioned calibration

Calibration을 `identity_id` SHA-256으로 fit/safety에 결정론적으로 분할합니다. FIQA cutpoint는 fit subset에서만 정하며, Low/High threshold는 group별 non-mated score에서 산출한 뒤 표본수 기반 shrinkage와 safety 상한을 적용합니다. Test에서는 고정된 cutpoint/threshold를 한 번만 적용합니다.

In [8]:
CALIBRATION_OUTPUT_DIR = (
    RESULT_ROOT / 'global_vs_fiqa' / str(source_manifest['run_id'])
    / selected_spec.model_uid / f'{COMPRESSION_PROFILE}__{SEARCH_MODE}'
)
comparison = None
calibration_stage_action = 'not_run'

if RUN_THRESHOLD_CALIBRATION:
    if CALIBRATION_OUTPUT_DIR.exists() and not OVERWRITE_OUTPUTS:
        comparison = load_calibration_comparison_artifact(CALIBRATION_OUTPUT_DIR)
        calibration_stage_action = 'reused_verified'
        display(Markdown(f'완료된 calibration artifact를 검증 후 재사용합니다: `{CALIBRATION_OUTPUT_DIR}`'))
    else:
        if calibration_with_fiqa is None or test_with_fiqa is None:
            raise RuntimeError('먼저 FIQA 및 condition score artifact를 준비해야 합니다.')
        in_memory_comparison = run_global_vs_fiqa_calibration(
            calibration_with_fiqa,
            test_with_fiqa,
            target_fpirs=TARGET_FPIRS,
            bin_count=QUALITY_BIN_COUNT,
            shrinkage_strength=SHRINKAGE_STRENGTH,
            minimum_group_non_mated=MINIMUM_GROUP_NON_MATED,
            safety_fraction=SAFETY_FRACTION,
            partition_seed=PARTITION_SEED,
            condition_manifest=condition_tables.manifest,
            fiqa_manifest=fiqa_artifact.manifest,
        )
        comparison = (
            write_calibration_comparison_artifact(
                CALIBRATION_OUTPUT_DIR, in_memory_comparison, overwrite=OVERWRITE_OUTPUTS
            )
            if WRITE_CALIBRATION_ARTIFACT
            else in_memory_comparison
        )
        calibration_stage_action = (
            'computed_written' if WRITE_CALIBRATION_ARTIFACT else 'computed_in_memory'
        )
else:
    if CALIBRATION_OUTPUT_DIR.is_dir():
        comparison = load_calibration_comparison_artifact(CALIBRATION_OUTPUT_DIR)
        calibration_stage_action = 'loaded_verified'

if comparison is not None:
    expected_comparison = {
        'condition_uid': condition_tables.condition_uid if condition_tables is not None else None,
        'fiqa_uid': fiqa_artifact.manifest['fiqa_uid'] if fiqa_artifact is not None else None,
        'score_space': condition_tables.manifest['score_space'] if condition_tables is not None else None,
    }
    mismatches = {
        key: (comparison.manifest.get(key), expected)
        for key, expected in expected_comparison.items()
        if expected is None or comparison.manifest.get(key) != expected
    }
    if mismatches:
        raise ValueError(f'calibration artifact가 현재 입력 lineage와 다릅니다: {mismatches}')

if comparison is None:
    display(Markdown(f'calibration comparison 대기 중: `{CALIBRATION_OUTPUT_DIR}`'))
else:
    result_columns = [
        'target_fpir', 'method', 'realized_fpir',
        'fpir_wilson95_low', 'fpir_wilson95_high', 'tpir_at_rank_k',
        'tpir_at_rank_k_wilson95_low', 'tpir_at_rank_k_wilson95_high',
        'target_met_on_test', 'target_met_by_wilson_upper',
    ]
    display(
        comparison.method_summary[result_columns]
        .sort_values(['target_fpir', 'method']).reset_index(drop=True)
    )
    display(comparison.paired_comparisons)

,target_fpir,method,realized_fpir,fpir_wilson95_low,fpir_wilson95_high,tpir_at_rank_k,tpir_at_rank_k_wilson95_low,tpir_at_rank_k_wilson95_high,target_met_on_test,target_met_by_wilson_upper
0,0.01,fiqa_2bin_conservative_shrunk_safe,0.014039,0.013393,0.014715,0.002698,0.002314,0.003144,False,False
1,0.01,global_empirical,0.014803,0.014139,0.015496,0.002880,0.002483,0.003340,False,False
2,0.01,global_safe,0.014071,0.013425,0.014749,0.002731,0.002345,0.003180,False,False
3,0.05,fiqa_2bin_conservative_shrunk_safe,0.051858,0.050627,0.053118,0.012164,0.011321,0.013070,False,False
4,0.05,global_empirical,0.053148,0.051902,0.054422,0.012164,0.011321,0.013070,False,False
5,0.05,global_safe,0.052688,0.051447,0.053957,0.012032,0.011193,0.012933,False,False
6,0.10,fiqa_2bin_conservative_shrunk_safe,0.093695,0.092070,0.095344,0.021399,0.020275,0.022584,True,True
7,0.10,global_empirical,0.095732,0.094092,0.097397,0.020853,0.019744,0.022023,True,True
8,0.10,global_safe,0.095379,0.093741,0.097041,0.020803,0.019695,0.021972,True,True
9,0.20,fiqa_2bin_conservative_shrunk_safe,0.180111,0.177962,0.182280,0.037304,0.035822,0.038844,True,True


,reference_method,candidate_method,target_fpir,score_space,rank_k,metric,reference_successes,candidate_successes,both_successes,total,candidate_minus_reference,paired_bootstrap95_low,paired_bootstrap95_high,confidence_interval_method,resamples,random_seed
0,global_empirical,fiqa_2bin_conservative_shrunk_safe,0.01,negative_squared_l2_adc,20,fpir,1802,1709,1641,121736,-0.000764,-0.001002,-0.000534,paired_nonparametric_bootstrap_percentile,2000,42
1,global_empirical,fiqa_2bin_conservative_shrunk_safe,0.01,negative_squared_l2_adc,20,tpir_at_rank_k,174,163,158,60423,-0.000182,-0.000331,-0.000033,paired_nonparametric_bootstrap_percentile,2000,42
2,global_safe,fiqa_2bin_conservative_shrunk_safe,0.01,negative_squared_l2_adc,20,fpir,1713,1709,1589,121736,-0.000033,-0.000279,0.000214,paired_nonparametric_bootstrap_percentile,2000,42
3,global_safe,fiqa_2bin_conservative_shrunk_safe,0.01,negative_squared_l2_adc,20,tpir_at_rank_k,165,163,152,60423,-0.000033,-0.000199,0.000132,paired_nonparametric_bootstrap_percentile,2000,42
4,global_empirical,fiqa_2bin_conservative_shrunk_safe,0.05,negative_squared_l2_adc,20,fpir,6470,6313,6216,121736,-0.001290,-0.001585,-0.001002,paired_nonparametric_bootstrap_percentile,2000,42
5,global_empirical,fiqa_2bin_conservative_shrunk_safe,0.05,negative_squared_l2_adc,20,tpir_at_rank_k,735,735,719,60423,0.000000,-0.000199,0.000182,paired_nonparametric_bootstrap_percentile,2000,42
6,global_safe,fiqa_2bin_conservative_shrunk_safe,0.05,negative_squared_l2_adc,20,fpir,6414,6313,6202,121736,-0.000830,-0.001125,-0.000534,paired_nonparametric_bootstrap_percentile,2000,42
7,global_safe,fiqa_2bin_conservative_shrunk_safe,0.05,negative_squared_l2_adc,20,tpir_at_rank_k,727,735,715,60423,0.000132,-0.000050,0.000314,paired_nonparametric_bootstrap_percentile,2000,42
8,global_empirical,fiqa_2bin_conservative_shrunk_safe,0.10,negative_squared_l2_adc,20,fpir,11654,11406,10688,121736,-0.002037,-0.002695,-0.001339,paired_nonparametric_bootstrap_percentile,2000,42
9,global_empirical,fiqa_2bin_conservative_shrunk_safe,0.10,negative_squared_l2_adc,20,tpir_at_rank_k,1260,1293,1193,60423,0.000546,0.000132,0.000960,paired_nonparametric_bootstrap_percentile,2000,42


## 9. Saliency 1차 목적 — 기존 압축/retrieval 진단 유지

이 셀은 새 threshold를 학습하지 않습니다. 기존 ArcFace/SurvFace 결과에서 PQ m128 ADC 조건의 공간적 saliency feature와 embedding distortion, score/rank 변화, crossing 간 연관 결과를 읽습니다. Spearman rho는 보조 진단이며 임의 threshold 가중식으로 바꾸지 않습니다.

In [9]:
primary_saliency_views = {}
if RUN_PRIMARY_SALIENCY_SUMMARY:
    primary_saliency = load_saliency_primary_diagnostics(SOURCE_RUN_DIR)
    for name, frame in primary_saliency.items():
        mask = pd.Series(True, index=frame.index)
        if 'compression_profile' in frame:
            mask &= frame['compression_profile'].astype(str).eq(COMPRESSION_PROFILE)
        if 'search_mode' in frame:
            mask &= frame['search_mode'].astype(str).eq(SEARCH_MODE)
        selected = frame.loc[mask].copy()
        preferred = [
            'analysis_scope', 'analysis_tier', 'compression_profile', 'search_mode',
            'target_fpir', 'threshold_policy', 'is_mated', 'saliency_feature',
            'instability_predictor', 'sensitivity_metric', 'event_metric',
            'sample_count', 'paired_query_count', 'identity_count', 'event_count',
            'event_rate', 'spearman_rho', 'bootstrap_ci_low', 'bootstrap_ci_high',
            'frozen_event_count', 'frozen_event_rate', 'recalibrated_event_count',
            'recalibrated_event_rate', 'recalibrated_minus_frozen_rate',
            'resolved_event_count', 'introduced_event_count',
            'frozen_spearman_rho', 'recalibrated_spearman_rho',
            'recalibrated_minus_frozen_rho', 'paired_bootstrap_ci_low',
            'paired_bootstrap_ci_high', 'event_support_eligible',
            'association_status',
        ]
        columns = [column for column in preferred if column in selected]
        primary_saliency_views[name] = selected[columns].reset_index(drop=True)
        display(Markdown(f'**{name}** — {len(selected):,} rows'))
        if (
            selected.empty
            and SEARCH_MODE == 'pq_adc_exhaustive'
            and name in {'threshold_policy', 'threshold_policy_rho'}
        ):
            display(Markdown('PQ ADC score-space에는 frozen-origin 대 recalibrated threshold policy 비교가 적용되지 않습니다(not applicable).'))
            continue
        preview = primary_saliency_views[name]
        if 'saliency_feature' in preview and not preview.empty:
            preview = (
                preview.sort_values('saliency_feature')
                .groupby('saliency_feature', sort=True, group_keys=False)
                .head(1)
                .reset_index(drop=True)
            )
        display(preview.head(30))
else:
    display(Markdown('`RUN_PRIMARY_SALIENCY_SUMMARY=False`: 기존 saliency 진단 로드를 건너뜁니다.'))

**geometry** — 57 rows

,analysis_scope,compression_profile,saliency_feature,sensitivity_metric,sample_count,identity_count,event_count,event_rate,spearman_rho,bootstrap_ci_low,bootstrap_ci_high,event_support_eligible,association_status
0,geometry,pq_512_m128_b8,face_attention,angular_error_rad,182159,124736,NaN,NaN,0.236145,0.226482,0.245894,NaN,continuous_outcome
1,geometry,pq_512_m128_b8,jaw_attention,angular_error_rad,182159,124736,NaN,NaN,0.156815,0.149698,0.164742,NaN,continuous_outcome
2,geometry,pq_512_m128_b8,left_cheek_attention,angular_error_rad,180861,123641,NaN,NaN,0.057545,0.051837,0.064117,NaN,continuous_outcome
3,geometry,pq_512_m128_b8,left_eye_attention,angular_error_rad,182159,124736,NaN,NaN,-0.081338,-0.088644,-0.074508,NaN,continuous_outcome
4,geometry,pq_512_m128_b8,left_right_asymmetry,angular_error_rad,182159,124736,NaN,NaN,0.275681,0.267836,0.283606,NaN,continuous_outcome
5,geometry,pq_512_m128_b8,maximum_region_concentration,angular_error_rad,182159,124736,NaN,NaN,0.074095,0.066954,0.081135,NaN,continuous_outcome
6,geometry,pq_512_m128_b8,mouth_attention,angular_error_rad,182159,124736,NaN,NaN,0.172042,0.164882,0.178970,NaN,continuous_outcome
7,geometry,pq_512_m128_b8,nose_attention,angular_error_rad,182159,124736,NaN,NaN,0.071563,0.065202,0.079534,NaN,continuous_outcome
8,geometry,pq_512_m128_b8,outside_face_attention,angular_error_rad,182159,124736,NaN,NaN,-0.236145,-0.245894,-0.226482,NaN,continuous_outcome
9,geometry,pq_512_m128_b8,quadrant_bottom_left,angular_error_rad,182159,124736,NaN,NaN,0.017621,0.010091,0.025272,NaN,continuous_outcome


**retrieval** — 3,040 rows

,analysis_scope,analysis_tier,compression_profile,search_mode,target_fpir,threshold_policy,is_mated,saliency_feature,sensitivity_metric,sample_count,identity_count,event_count,event_rate,spearman_rho,bootstrap_ci_low,bootstrap_ci_high,event_support_eligible,association_status
0,retrieval,exploratory,pq_512_m128_b8,pq_adc_exhaustive,0.01,recalibrated_compressed,False,face_attention,absolute_origin_winner_score_drift,0,0,NaN,NaN,NaN,NaN,NaN,NaN,continuous_outcome
1,retrieval,exploratory,pq_512_m128_b8,pq_adc_exhaustive,0.01,recalibrated_compressed,False,jaw_attention,absolute_origin_winner_score_drift,0,0,NaN,NaN,NaN,NaN,NaN,NaN,continuous_outcome
2,retrieval,exploratory,pq_512_m128_b8,pq_adc_exhaustive,0.01,recalibrated_compressed,False,left_cheek_attention,absolute_origin_winner_score_drift,0,0,NaN,NaN,NaN,NaN,NaN,NaN,continuous_outcome
3,retrieval,exploratory,pq_512_m128_b8,pq_adc_exhaustive,0.01,recalibrated_compressed,False,left_eye_attention,absolute_origin_winner_score_drift,0,0,NaN,NaN,NaN,NaN,NaN,NaN,continuous_outcome
4,retrieval,exploratory,pq_512_m128_b8,pq_adc_exhaustive,0.01,recalibrated_compressed,False,left_right_asymmetry,absolute_origin_winner_score_drift,0,0,NaN,NaN,NaN,NaN,NaN,NaN,continuous_outcome
5,retrieval,exploratory,pq_512_m128_b8,pq_adc_exhaustive,0.01,recalibrated_compressed,False,maximum_region_concentration,absolute_origin_winner_score_drift,0,0,NaN,NaN,NaN,NaN,NaN,NaN,continuous_outcome
6,retrieval,exploratory,pq_512_m128_b8,pq_adc_exhaustive,0.01,recalibrated_compressed,False,mouth_attention,absolute_origin_winner_score_drift,0,0,NaN,NaN,NaN,NaN,NaN,NaN,continuous_outcome
7,retrieval,exploratory,pq_512_m128_b8,pq_adc_exhaustive,0.01,recalibrated_compressed,False,nose_attention,absolute_origin_winner_score_drift,0,0,NaN,NaN,NaN,NaN,NaN,NaN,continuous_outcome
8,retrieval,exploratory,pq_512_m128_b8,pq_adc_exhaustive,0.01,recalibrated_compressed,False,outside_face_attention,absolute_origin_winner_score_drift,0,0,NaN,NaN,NaN,NaN,NaN,NaN,continuous_outcome
9,retrieval,exploratory,pq_512_m128_b8,pq_adc_exhaustive,0.01,recalibrated_compressed,False,quadrant_bottom_left,absolute_origin_winner_score_drift,0,0,NaN,NaN,NaN,NaN,NaN,NaN,continuous_outcome


**threshold_instability** — 450 rows

,analysis_scope,analysis_tier,compression_profile,search_mode,target_fpir,threshold_policy,is_mated,instability_predictor,sensitivity_metric,sample_count,identity_count,event_count,event_rate,spearman_rho,bootstrap_ci_low,bootstrap_ci_high,event_support_eligible,association_status
0,threshold_instability,exploratory,pq_512_m128_b8,pq_adc_exhaustive,0.01,recalibrated_compressed,False,absolute_origin_winner_score_drift,accept_to_reject_crossing,0,0,0,NaN,NaN,NaN,NaN,False,insufficient_event_support
1,threshold_instability,exploratory,pq_512_m128_b8,pq_adc_exhaustive,0.01,recalibrated_compressed,False,absolute_origin_winner_score_drift,false_accept_gain,0,0,0,NaN,NaN,NaN,NaN,False,insufficient_event_support
2,threshold_instability,exploratory,pq_512_m128_b8,pq_adc_exhaustive,0.01,recalibrated_compressed,False,absolute_origin_winner_score_drift,false_accept_loss,0,0,0,NaN,NaN,NaN,NaN,False,insufficient_event_support
3,threshold_instability,exploratory,pq_512_m128_b8,pq_adc_exhaustive,0.01,recalibrated_compressed,False,absolute_origin_winner_score_drift,reject_to_accept_crossing,0,0,0,NaN,NaN,NaN,NaN,False,insufficient_event_support
4,threshold_instability,exploratory,pq_512_m128_b8,pq_adc_exhaustive,0.01,recalibrated_compressed,False,absolute_origin_winner_score_drift,threshold_crossing,0,0,0,NaN,NaN,NaN,NaN,False,insufficient_event_support
5,threshold_instability,exploratory,pq_512_m128_b8,pq_adc_exhaustive,0.01,recalibrated_compressed,False,absolute_origin_winner_score_drift,tpir_at_rank_k_gain,0,0,0,NaN,NaN,NaN,NaN,False,insufficient_event_support
6,threshold_instability,exploratory,pq_512_m128_b8,pq_adc_exhaustive,0.01,recalibrated_compressed,False,absolute_origin_winner_score_drift,tpir_at_rank_k_loss,0,0,0,NaN,NaN,NaN,NaN,False,insufficient_event_support
7,threshold_instability,exploratory,pq_512_m128_b8,pq_adc_exhaustive,0.01,recalibrated_compressed,False,absolute_origin_winner_score_drift,tpir_rank_loss,0,0,0,NaN,NaN,NaN,NaN,False,insufficient_event_support
8,threshold_instability,exploratory,pq_512_m128_b8,pq_adc_exhaustive,0.01,recalibrated_compressed,False,absolute_origin_winner_score_drift,tpir_threshold_loss,0,0,0,NaN,NaN,NaN,NaN,False,insufficient_event_support
9,threshold_instability,exploratory,pq_512_m128_b8,pq_adc_exhaustive,0.01,recalibrated_compressed,False,absolute_threshold_margin_shift,accept_to_reject_crossing,0,0,0,NaN,NaN,NaN,NaN,False,insufficient_event_support


**threshold_policy** — 0 rows

PQ ADC score-space에는 frozen-origin 대 recalibrated threshold policy 비교가 적용되지 않습니다(not applicable).

**threshold_policy_rho** — 0 rows

PQ ADC score-space에는 frozen-origin 대 recalibrated threshold policy 비교가 적용되지 않습니다(not applicable).

## 10. Saliency 2차 목적 — FIQA 이후 추가정보 검증 readiness gate

사전 지정 feature는 `outside_face_attention`, `saliency_entropy` 두 개뿐입니다. `Random`은 threshold feature가 아니라 faithfulness negative control로만 사용합니다. 강한 reliability gate는 identity-cluster bootstrap에서 `High−Low`와 `High−Random`의 CI 하한이 모두 0보다 클 때만 통과합니다. 하나라도 실패하면 gated High/Low contrast는 0입니다. 이 gate와 별개로 calibration/test 모두에서 동일 target의 saliency coverage가 95% 이상이어야 다음 노트북(`01_saliency_incremental_threshold_calibration.ipynb`, 향후 작성 대상)으로 진행할 수 있습니다. 현재 artifact는 test-only이므로 실제 FIQA+Saliency calibration은 차단됩니다.

In [10]:
saliency_readiness = None
faithfulness_reliability = None
faithfulness_summary = pd.DataFrame()
saliency_correction_enabled = False
saliency_reliability_weight = 0.0
SALIENCY_FEATURE_PATH = (
    SOURCE_RUN_DIR / 'artifacts' / 'step2_workflow'
    / 'saliency_population' / 'saliency_features.csv'
)
if RUN_SALIENCY_READINESS_CHECK:
    if condition_tables is None:
        raise RuntimeError('readiness 계산에는 condition score artifact가 먼저 필요합니다.')
    faithfulness_maximum_samples = resolve_common_faithfulness_maximum_samples(
        PROJECT_ROOT,
        datasets=('survface',),
        run_ids={'survface': str(source_manifest['run_id'])},
    )
    faithfulness_artifacts = load_selected_faithfulness_artifacts(
        PROJECT_ROOT,
        datasets=('survface',),
        model_uids={'survface': source_model_uid},
        run_ids={'survface': str(source_manifest['run_id'])},
        maximum_samples=faithfulness_maximum_samples,
    )
    faithfulness_summary = faithfulness_artifacts.summary
    faithfulness_reliability = assess_saliency_faithfulness_reliability(
        faithfulness_summary, group='all'
    )
    display(
        faithfulness_summary.loc[
            faithfulness_summary['group'].astype(str).eq('all'),
            ['metric', 'sample_count', 'identity_count', 'mean', 'mean_ci_lower', 'mean_ci_upper'],
        ].reset_index(drop=True)
    )
    display(pd.DataFrame([faithfulness_reliability.as_dict()]))
    saliency_features = pd.read_csv(
        SALIENCY_FEATURE_PATH,
        usecols=[
            'sample_id', 'saliency_target_name', 'heatmap_available',
            'gradcam_valid_heatmap', 'outside_face_attention', 'saliency_entropy',
        ],
        low_memory=False,
    )
    saliency_readiness = assess_saliency_incremental_readiness(
        condition_tables.calibration,
        condition_tables.test,
        saliency_features,
        requested_features=('outside_face_attention', 'saliency_entropy'),
        minimum_coverage=0.95,
    )
    display(pd.DataFrame([saliency_readiness.as_dict()]))
    saliency_correction_enabled = bool(
        saliency_readiness.secondary_calibration_supported
        and faithfulness_reliability.strong_faithfulness_pass
    )
    saliency_reliability_weight = (
        faithfulness_reliability.gated_high_low_contrast
        if saliency_correction_enabled
        else 0.0
    )
    if not faithfulness_reliability.strong_faithfulness_pass:
        display(Markdown('**차단됨:** High가 Low와 Random을 모두 이기지 못했습니다. Random은 negative control로만 보고하며 saliency correction은 0입니다.'))
    if not saliency_readiness.secondary_calibration_supported:
        display(Markdown('**차단됨:** calibration saliency 없이 FIQA+Saliency threshold를 fit하면 test leakage가 되므로 saliency correction은 0입니다.'))
else:
    display(Markdown('`RUN_SALIENCY_READINESS_CHECK=False`: faithfulness/Random gate와 대용량 saliency feature 파일을 읽지 않습니다. 2차 calibration은 차단 상태입니다.'))

,metric,sample_count,identity_count,mean,mean_ci_lower,mean_ci_upper
0,high_saliency_occlusion_score_drop,182159,124736,0.089815,0.088817,0.090734
1,low_saliency_occlusion_score_drop,182159,124736,0.032345,0.031900,0.032785
2,random_occlusion_score_drop,182159,124736,0.115092,0.113999,0.116185
3,faithfulness_gain_over_low_saliency,182159,124736,0.057470,0.056744,0.058226
4,faithfulness_gain_over_random,182159,124736,-0.025278,-0.026108,-0.024425


,status,strong_faithfulness_pass,high_over_low_mean,high_over_low_ci_lower,high_over_low_ci_upper,high_over_random_mean,high_over_random_ci_lower,high_over_random_ci_upper,gated_high_low_contrast,group,random_control_role,random_is_threshold_feature,reasons
0,blocked,False,0.05747,0.056744,0.058226,-0.025278,-0.026108,-0.024425,0.0,all,faithfulness_negative_control_only,False,[High does not exceed Random with a strictly p...


,status,primary_analysis_supported,secondary_calibration_supported,calibration_coverage,test_coverage,reasons,saliency_target_name,requested_features
0,blocked,True,False,0.0,1.0,[calibration saliency coverage is below the pr...,origin_top1_gallery_cosine,"[outside_face_attention, saliency_entropy]"


**차단됨:** High가 Low와 Random을 모두 이기지 못했습니다. Random은 negative control로만 보고하며 saliency correction은 0입니다.

**차단됨:** calibration saliency 없이 FIQA+Saliency threshold를 fit하면 test leakage가 되므로 saliency correction은 0입니다.

In [11]:
# 11. 현재 상태 요약 — 계산 완료와 미실행을 명확히 구분
status_rows = [
    {'stage': 'checkpoint_preflight', 'status': 'validated', 'artifact': str(FIQA_CHECKPOINTS[FIQA_VARIANT])},
    {'stage': 'fiqa_scores', 'status': 'available' if fiqa_artifact is not None else 'not_run', 'action': fiqa_stage_action, 'artifact': str(FIQA_OUTPUT_DIR)},
    {'stage': 'calibration_score_replay', 'status': 'available' if condition_tables is not None else 'not_run', 'action': condition_stage_action, 'artifact': str(CONDITION_OUTPUT_DIR)},
    {'stage': 'global_vs_fiqa', 'status': 'available' if comparison is not None else 'not_run', 'action': calibration_stage_action, 'artifact': str(CALIBRATION_OUTPUT_DIR)},
    {'stage': 'saliency_primary_analysis', 'status': 'available' if primary_saliency_views else 'not_loaded', 'artifact': str(SOURCE_RUN_DIR / 'artifacts' / 'step2_workflow')},
    {
        'stage': 'saliency_faithfulness_random_control',
        'status': faithfulness_reliability.status if faithfulness_reliability is not None else 'not_checked',
        'action': f'gated_contrast={saliency_reliability_weight:.12g}',
        'artifact': str(faithfulness_artifacts.roots['survface']) if faithfulness_reliability is not None else 'not_loaded',
    },
    {
        'stage': 'fiqa_plus_saliency',
        'status': 'ready' if saliency_correction_enabled else 'blocked',
        'action': 'enabled' if saliency_correction_enabled else 'correction_zero',
        'artifact': 'not_created',
    },
]
status_table = pd.DataFrame(status_rows)
display(status_table)

,stage,status,artifact,action
0,checkpoint_preflight,validated,C:\ronbun\models\fiqa\CR-FIQA(L).pth,NaN
1,fiqa_scores,available,C:\ronbun\results\calibration\fiqa_scores\surv...,computed_written
2,calibration_score_replay,available,C:\ronbun\results\calibration\condition_scores...,loaded_verified
3,global_vs_fiqa,available,C:\ronbun\results\calibration\global_vs_fiqa\2...,computed_written
4,saliency_primary_analysis,available,C:\ronbun\runs\survface_20260902\20260902-R001...,NaN
5,saliency_faithfulness_random_control,blocked,C:\ronbun\results\paper\survface\20260902-R001...,gated_contrast=0
6,fiqa_plus_saliency,blocked,not_created,correction_zero


## 12. 해석 규칙

- 결론은 `realized_fpir`, Wilson 95% CI, `tpir_at_rank_k`(현재 Rank-20), `target_met_on_test`, paired bootstrap 차이를 함께 봅니다.
- `global_safe`와 `shrunk_safe`는 held-out 경험적 보수화이며 formal FPIR guarantee가 아닙니다.
- FIQA가 아주 조금 좋아졌다는 이유만으로 채택하지 않습니다. 사전 지정한 여러 target FPIR에서 방향이 재현되고, paired CI와 TPIR 손실까지 검토해야 합니다.
- FIQA가 Global보다 낫지 않아도 saliency의 1차 연구 질문은 독립적으로 유지됩니다.
- Random은 calibration/threshold feature가 아니라 필수 faithfulness negative control입니다. `High−Low`와 `High−Random` paired CI 하한이 모두 양수일 때만 strong faithfulness를 통과합니다.
- strong faithfulness 또는 calibration/test saliency coverage 중 하나라도 실패하면 `saliency_correction_enabled=False`, `saliency_reliability_weight=0`으로 유지합니다.
- FIQA+Saliency는 calibration saliency를 별도로 생성한 뒤에만 시험하며, test 기반 feature 선택이나 threshold 재조정은 금지합니다.
- 이 노트북에서 생성한 compact artifact만 이후 `00_cross_dataset_results.ipynb`의 입력 후보가 됩니다. 공통 보고 노트북 연결은 결과가 실제로 생성·검증된 뒤 별도 변경으로 수행합니다.